*Instructor / solutions notebook — every exercise is filled in and commented. Runs top to bottom.*

# Reinforcement Learning From Scratch: Landing on the Moon

### A half-day workshop

<img src="https://gymnasium.farama.org/_images/lunar_lander.gif" width="400">

---

By the end of this notebook you should be able to:

1. Explain what reinforcement learning is and how it differs from supervised learning
2. Implement tabular Q-learning from scratch and use it to solve a small environment
3. Explain why a lookup table stops working as the state space grows, and build a Deep Q-Network to fix it
4. Train an agent that lands a spacecraft on the moon

**How this notebook is organized**

- **Exercise** blocks contain code you write yourself. They won't run until you fill them in.
- **Question** blocks are things to answer before you keep scrolling. Guessing wrong and then finding out why is most of how this material actually sticks.
- **Look it up** blocks point you at documentation instead of handing you the answer. Reading documentation is most of what doing RL looks like day to day, so treat this as practice.

One ground rule for today: when something breaks, read the error message, form a hypothesis about what's wrong, and test it before asking anyone for the fix. Debugging RL code is its own skill, and this notebook is where you start building it.

---

### Before you start

Go to **Runtime > Change runtime type > T4 GPU** if one is available — CPU works too, just slower — then run the cell below. It takes about two minutes.

In [ ]:
#@title Install dependencies (run me first — takes ~2 min) { display-mode: "form" }
# swig must be installed BEFORE box2d-py, because box2d-py's build uses it.
!pip install -q swig
!pip install -q "gymnasium[box2d]==1.3.0" imageio imageio-ffmpeg
print("\n Done. If you see errors above about box2d-py wheels, re-run this cell once.")

In [ ]:
import os
# Box2D renders through pygame, which wants a display. Colab has none, so we
# point SDL at a "dummy" driver: it draws into memory and we grab the pixels.
os.environ["SDL_VIDEODRIVER"] = "dummy"

import random, math, time
from collections import deque, namedtuple

import numpy as np
import gymnasium as gym
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import imageio
from IPython.display import HTML, display
import base64

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("gymnasium:", gym.__version__)
print("torch:", torch.__version__)
print("device:", DEVICE)

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

set_seed()

In [ ]:
#@title Helper: record an episode as video { display-mode: "form" }
def record_episode(env_id, policy_fn, filename="agent.mp4", max_steps=1000, fps=50, **env_kwargs):
    # policy_fn(observation) -> action.  Returns total reward.
    env = gym.make(env_id, render_mode="rgb_array", **env_kwargs)
    obs, _ = env.reset(seed=random.randint(0, 10_000))
    frames, total = [], 0.0
    for _ in range(max_steps):
        frames.append(env.render())
        obs, r, terminated, truncated, _ = env.step(policy_fn(obs))
        total += r
        if terminated or truncated:
            break
    env.close()
    imageio.mimsave(filename, frames, fps=fps)
    b64 = base64.b64encode(open(filename, "rb").read()).decode()
    display(HTML(f'<video width="480" controls autoplay loop><source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'))
    return total

def plot_scores(scores, window=100, title="Training progress"):
    plt.figure(figsize=(10, 4))
    plt.plot(scores, alpha=0.3, label="episode reward")
    if len(scores) >= window:
        smooth = np.convolve(scores, np.ones(window) / window, mode="valid")
        plt.plot(range(window - 1, len(scores)), smooth, lw=2, label=f"{window}-ep moving avg")
    plt.axhline(200, color="green", ls="--", label="'solved' (200)")
    plt.xlabel("Episode"); plt.ylabel("Reward"); plt.title(title)
    plt.legend(); plt.grid(alpha=0.3); plt.show()

print("Helpers loaded.")

---
# Part 1 — What reinforcement learning actually is

## 1.1 The one-sentence version

An **agent** takes **actions** in an **environment**, receives **rewards**, and learns a **policy** that maximizes total reward over time.

That's the whole idea. Everything else in this notebook is engineering detail built on top of it.

## 1.2 How this differs from the ML you've already seen

| | Supervised learning | Reinforcement learning |
|---|---|---|
| Training signal | A correct label for every input | A scalar reward, often delayed, never a "correct answer" |
| Data source | A fixed dataset | The agent generates its own data by acting |
| i.i.d. assumption | Usually holds | Badly violated — your data depends on your current policy |
| Typical failure mode | Overfitting | Never finding decent behavior at all |

That last row is why RL is hard. A supervised dataset sits still while you train on it. In RL, improving your policy changes the distribution of data you collect next, which changes what you learn, which changes your policy again. It's a feedback loop, and feedback loops can diverge.

**Question.** A chess engine gets a reward of +1 for winning, 0 for a draw, and -1 for losing — and nothing for the forty moves in between. If it loses, which of those forty moves was the mistake?

This is the **credit assignment problem**. Keep it in mind; the discount factor γ, introduced below, is our partial answer to it.

## 1.3 Vocabulary you'll need for the rest of this notebook

RL problems are formally modeled as a **Markov Decision Process (MDP)**:

- **State** $s$ — what the agent observes. In LunarLander this is 8 numbers: position, velocity, angle, and leg contacts.
- **Action** $a$ — what the agent can do. LunarLander has 4 discrete choices: do nothing, or fire the left, main, or right engine.
- **Reward** $r$ — a scalar signal returned after each action.
- **Policy** $\pi(a|s)$ — the agent's strategy, a mapping from states to actions.
- **Return** $G_t = r_t + \gamma r_{t+1} + \gamma^2 r_{t+2} + \dots$ — total discounted future reward.
- **Discount factor** $\gamma \in [0, 1)$ — how much weight we give future reward relative to immediate reward.

The **Markov property** is the assumption that the current state contains everything needed to decide what to do next — the history before it adds nothing. This is a strong assumption, and it's often false in the real world, which is part of what "partially observable" means.

**Question.** With γ = 0, the agent only cares about immediate reward. With γ = 0.99, a reward 100 steps away is still worth 0.99^100 ≈ 0.37 of its face value. What breaks if you set γ = 1.0 in a task that never terminates? Try writing out the sum $r + r + r + \dots$ to see it.

## 1.4 The single most important object: the Q-function

$$Q^\pi(s, a) = \mathbb{E}\left[\, G_t \mid s_t = s,\ a_t = a,\ \text{then follow } \pi \,\right]$$

In words: if I'm in state $s$ and take action $a$, how much total reward should I expect from here on, assuming I follow policy $\pi$ afterward?

Here's why this matters: if you had the optimal Q-function $Q^*$, choosing the best action is trivial — just take `argmax_a Q*(s, a)`. A perfect Q-function is a perfect policy. So the whole problem reduces to estimating Q.

The recursive relationship that makes this estimable is the **Bellman optimality equation**:

$$Q^*(s, a) = \mathbb{E}\Big[\, r + \gamma \max_{a'} Q^*(s', a') \,\Big]$$

Read it as: the value of taking this action equals the reward you get right now, plus the discounted value of the best action available next. Every algorithm in this notebook is a way of forcing an estimate to satisfy this equation.

**Look it up.**
- [Hugging Face Deep RL Course, Unit 1](https://huggingface.co/learn/deep-rl-course/unit1/introduction) — a solid free introduction, worth doing after this workshop.
- [Sutton & Barto, *Reinforcement Learning: An Introduction*](http://incompleteideas.net/book/the-book-2nd.html) — the field's standard textbook, free online. Chapters 3 (MDPs) and 6 (TD learning) cover what we're doing today.
- [GeeksforGeeks — Markov Decision Process](https://www.geeksforgeeks.org/markov-decision-process/) — a quick reference.
- [Lilian Weng — A (Long) Peek into RL](https://lilianweng.github.io/posts/2018-02-19-rl-overview/) — dense but thorough.

Before moving on, work out the difference between $Q(s,a)$ and $V(s)$. They're closely related — what's the exact relationship under an optimal policy? You'll need this in about twenty minutes.

---
# Part 2 — Meet the environment

We'll use [Gymnasium](https://gymnasium.farama.org/), the maintained fork of OpenAI Gym. Every environment shares the same interface, so an algorithm written for one works on all of them with minimal changes.

The core loop is always:

```python
obs, info = env.reset(seed=0)          # start an episode
obs, reward, terminated, truncated, info = env.step(action)   # take one step
```

**Question.** Why are `terminated` and `truncated` two separate flags instead of one?

`terminated` means the episode genuinely ended — the lander crashed or landed. `truncated` means we cut the episode off artificially because it hit a step limit. Older versions of Gym combined these into a single `done` flag. Conflating them causes a real bug in value learning, which we'll run into in Part 5 — try to predict what it is before we get there. See the [Gymnasium migration guide](https://gymnasium.farama.org/introduction/migration_guide/) for more detail.

In [ ]:
env = gym.make("LunarLander-v3")

print("Observation space:", env.observation_space)
print("Action space:     ", env.action_space)
print()
obs, info = env.reset(seed=0)
print("A single observation:", np.round(obs, 3))
print()
print("The 8 numbers mean:")
for i, name in enumerate(["x position", "y position", "x velocity", "y velocity",
                          "angle", "angular velocity", "left leg contact", "right leg contact"]):
    print(f"  obs[{i}] = {obs[i]: .3f}   {name}")
env.close()

**Look it up — read the environment docs before writing any code.**

Open the [LunarLander documentation](https://gymnasium.farama.org/environments/box2d/lunar_lander/) and answer the following. Don't skip this — knowing the reward function is knowing the problem.

1. What are the four discrete actions, in order? Which integer corresponds to firing the main engine?
2. Write out the reward function. What do you get for moving toward the pad? For crashing? For landing? What does firing the main engine cost per frame?
3. What total episode reward counts as "solved"?
4. `obs[6]` and `obs[7]` are booleans, not floats. What do they represent?
5. There's a `continuous=True` option. What changes? Would our DQN still work with it? (More on this later.)

Write your answers below. You'll need question 2 in particular when debugging a strangely-behaving agent later on.

### Your answers

1. 
2. 
3. 
4. 
5. 

## 2.1 Baseline: a random agent

Always measure a baseline before optimizing anything. If you don't know what random behavior looks like, you can't tell whether your agent learned something or just got lucky.

In [ ]:
env_probe = gym.make("LunarLander-v3")
random_scores = []
for ep in range(20):
    obs, _ = env_probe.reset(seed=ep)
    total, done = 0.0, False
    while not done:
        obs, r, term, trunc, _ = env_probe.step(env_probe.action_space.sample())
        total += r
        done = term or trunc
    random_scores.append(total)

RANDOM_BASELINE = float(np.mean(random_scores))   # remember this for later
print(f"Random agent over 20 episodes: mean {RANDOM_BASELINE:.1f}, std {np.std(random_scores):.1f}")
print(f"Best random episode: {max(random_scores):.1f}   |   'Solved' threshold: 200")

In [ ]:
# Watch it flail. This is our starting point.
total = record_episode("LunarLander-v3", lambda o: env_probe.action_space.sample(), "random.mp4")
env_probe.close()
print(f"Episode reward: {total:.1f}")

**Question.** The random agent scores around -180. Where does that large negative number come from? Check your answer to question 2 above — two separate sources contribute to it. Name both.

---
# Part 3 — Tabular Q-learning: the whole idea, no neural networks yet

Before touching PyTorch, we'll implement RL in its simplest form. Strip away the deep learning and Q-learning is about fifteen lines of code. Understand those fifteen lines well and DQN is mostly bookkeeping layered on top.

## 3.1 The setup

When states and actions are both finite and small, we can store Q as a plain 2D array:

```
Q[state, action] -> expected return
```

We'll use [FrozenLake](https://gymnasium.farama.org/environments/toy_text/frozen_lake/): a 4x4 grid with 16 states and 4 actions. The goal is to walk from start to goal without falling in a hole. With `is_slippery=True`, the ice makes you slide — your chosen action only happens with probability 1/3, so the environment is stochastic. That turns out to matter a lot, as you'll see shortly.

The reward is +1 for reaching the goal and 0 for everything else. That's the entire signal.

## 3.2 The update rule

$$Q(s,a) \leftarrow Q(s,a) + \alpha \Big[\underbrace{r + \gamma \max_{a'} Q(s',a')}_{\text{TD target}} - \underbrace{Q(s,a)}_{\text{current estimate}}\Big]$$

Read this as: nudge my current estimate toward a better estimate, by a fraction $\alpha$.

The bracketed term is the **TD error** — the gap between what I predicted and what I now believe after seeing one real step of experience. Learning is the process of shrinking that gap.

The useful part is that $r + \gamma \max Q(s',a')$ is a better estimate than $Q(s,a)$ alone, because it contains one step of real, observed reward. This is bootstrapping: using our own guesses to improve our own guesses, anchored by real data trickling in from the environment.

## 3.3 Exploration versus exploitation

If the agent always takes `argmax Q`, it will lock onto the first mediocre strategy that works and never discover anything better. So we use ε-greedy: with probability ε, act randomly; otherwise, act greedily. Start ε high to explore, then decay it over time to exploit what's been learned.

**Question.** What happens if ε decays to 0 too quickly? What if it never decays at all? Both are failure modes — describe what each looks like in the reward curve.

**Look it up.** Look up optimistic initialization and Boltzmann (softmax) exploration, two alternatives to ε-greedy. When would you prefer each one?

### Solution 1 — Q-learning

Notice how little code this actually is. The `if terminated` guard on the TD target is the part people get wrong most often, so read that comment carefully.

In [ ]:
def epsilon_greedy(Q, state, epsilon, n_actions):
    if np.random.rand() < epsilon:
        return np.random.randint(n_actions)      # explore
    return int(np.argmax(Q[state]))              # exploit


def q_learning(env, n_episodes=20_000, alpha=0.1, gamma=0.99,
               eps_start=1.0, eps_end=0.01, eps_decay=0.9995):
    n_states  = env.observation_space.n
    n_actions = env.action_space.n
    Q = np.zeros((n_states, n_actions))
    epsilon = eps_start
    rewards = []

    for ep in range(n_episodes):
        state, _ = env.reset()
        total, done = 0.0, False

        while not done:
            action = epsilon_greedy(Q, state, epsilon, n_actions)
            next_state, reward, terminated, truncated, _ = env.step(action)

            # If the episode TERMINATED, the future is worth exactly nothing —
            # there is no s'. Bootstrapping past a terminal state is a classic bug
            # that makes values blow up.
            # Note we check `terminated`, NOT `terminated or truncated`: a truncated
            # episode was cut short artificially, the future still exists.
            if terminated:
                td_target = reward
            else:
                td_target = reward + gamma * np.max(Q[next_state])

            Q[state, action] += alpha * (td_target - Q[state, action])

            state = next_state
            total += reward
            done = terminated or truncated

        epsilon = max(eps_end, epsilon * eps_decay)
        rewards.append(total)

    return Q, rewards

In [ ]:
set_seed(0)
lake = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=True)
Q_table, fl_rewards = q_learning(lake, n_episodes=20_000)

window = 500
smooth = np.convolve(fl_rewards, np.ones(window) / window, mode="valid")
plt.figure(figsize=(10, 3.5))
plt.plot(smooth)
plt.xlabel("Episode"); plt.ylabel(f"Success rate ({window}-ep avg)")
plt.title("FrozenLake (slippery) — tabular Q-learning"); plt.grid(alpha=0.3); plt.show()

print(f"Success rate, last 1000 episodes: {np.mean(fl_rewards[-1000:]):.1%}")

In [ ]:
# The learned Q-table. Rows = the 16 grid cells, columns = actions.
np.set_printoptions(precision=3, suppress=True)
print("      LEFT   DOWN  RIGHT     UP")
for s in range(16):
    print(f"s={s:2d}  {Q_table[s]}")

arrows = ["<", "v", ">", "^"]
print("\nGreedy policy on the grid:")
desc = lake.unwrapped.desc.astype(str).flatten()
for r in range(4):
    row = ""
    for c in range(4):
        s = r * 4 + c
        row += ("  H  " if desc[s] == "H" else "  G  " if desc[s] == "G"
                else f"  {arrows[np.argmax(Q_table[s])]}  ")
    print(row)

**Question — this policy looks wrong. Why isn't it?**

Look at the arrows in the output above. In several cells, the "best" action points away from the goal, or straight into a wall.

The ice is slippery: your action only succeeds a third of the time, and otherwise you slide sideways. So the optimal strategy is often to aim at a wall, because bouncing off a wall is safe, while aiming across open ice next to a hole risks sliding straight into it.

The agent learned to exploit the physics of the environment. It found a policy a human would call strange, but it's actually correct given the reward it was optimizing.

This is worth sitting with: RL agents optimize the reward function you wrote, not the behavior you intended. When the two diverge, you get [reward hacking](https://openai.com/index/faulty-reward-functions/) — the boat-racing example in that post is a classic illustration.

**Try this** (pick at least two):

1. Set `is_slippery=False` and re-run. Does the policy become more intuitive?
2. Set `gamma=0.5`. What changes, and why? Think about how far the +1 signal has to propagate backward.
3. Set `eps_decay=0.99`, a much faster decay. What breaks?
4. Try the 8x8 map. Does 20,000 episodes still work?

---
# Part 4 — Why the table has to go

We now have a working RL algorithm. Point it at LunarLander.

Except we can't, not directly. `Q[state, action]` needs `state` to be an integer index, and LunarLander's state is 8 continuous floats.

The obvious fix is to discretize: chop each dimension into bins. Let's see what that costs.

In [ ]:
for bins in [5, 10, 20]:
    n_states = bins ** 8
    print(f"{bins:>3} bins per dimension -> {n_states:,} states x 4 actions "
          f"= {n_states*4:,} table entries ({n_states*4*8/1e9:.2f} GB as float64)")

print("\nAnd the real problem isn't memory:")
print("  * Each cell must be VISITED MANY TIMES to get a reliable estimate.")
print("  * A cell you have never visited has value 0.0 -- a pure guess.")
print("  * The table has NO NOTION that two nearby states are similar.")

This is the **curse of dimensionality**, and the third point above is the one that actually kills the approach.

A table treats the state (x=0.10, y=1.40, ...) and the state (x=0.11, y=1.40, ...) as completely unrelated entries. Learning about one teaches you nothing about the other, even though physically they're almost the same situation and the correct action is almost certainly identical.

What we need instead is generalization: a function that maps similar inputs to similar outputs, and interpolates sensibly between the states we've actually visited.

That's a function approximator. Specifically, a neural network.

$$Q(s, a) \quad\longrightarrow\quad Q(s, a; \theta)$$

Replace the lookup table with a network with parameters θ. Instead of writing into a table cell, we take a gradient step. This is deep Q-learning.

**Look it up.** Function approximation makes Q-learning unstable — it can and does diverge, unlike the tabular case, which has convergence guarantees. Search for "the deadly triad" in RL. What are its three ingredients, and which of them are we about to use? (All three, as it happens.) Everything in the next section exists to manage that instability.

---
# Part 5 — Deep Q-Networks

DQN is Q-learning plus a neural network plus two stabilization tricks. Those two tricks are the actual contribution of the [2015 Nature paper](https://www.nature.com/articles/nature14236) that kicked off the deep RL era, and both exist specifically to fight problems introduced by the network.

### Trick 1 — Experience replay

**The problem.** Consecutive transitions are heavily correlated — frames 1 and 2 of a descent look nearly identical. Gradient descent assumes i.i.d. samples, and feeding it a correlated stream causes it to overfit to whatever the agent happens to be doing right now, forgetting everything else it learned earlier ([catastrophic forgetting](https://en.wikipedia.org/wiki/Catastrophic_interference)).

**The fix.** Store every transition `(s, a, r, s', done)` in a large buffer, and train on random minibatches sampled from it. This breaks the correlation between consecutive samples and lets each experience be reused many times, which improves sample efficiency substantially.

### Trick 2 — Target network

**The problem.** Our loss looks like this:

$$L(\theta) = \Big( \underbrace{r + \gamma \max_{a'} Q(s', a'; \theta)}_{\text{target}} - Q(s, a; \theta) \Big)^2$$

The target depends on θ, the same parameters we're updating. So every gradient step moves the target we're chasing — like trying to hit a bullseye that jumps every time you draw the bow. This causes oscillation and, in the worst case, divergence.

**The fix.** Keep a second, slowly-updated copy of the network, θ⁻, used only for computing targets:

$$L(\theta) = \Big( r + \gamma \max_{a'} Q(s', a'; \theta^-) - Q(s, a; \theta) \Big)^2$$

Now the target stays fixed for a while. We sync θ⁻ ← θ periodically — either a hard copy every N steps, or a soft update θ⁻ ← τθ + (1-τ)θ⁻ with a small τ, which is what we'll use here.

**Question.** If a target network helps by keeping targets stale, why not just freeze it permanently? There's a real tradeoff here — name both sides of it.

**Look it up.**
- [HF Deep RL Course, Unit 3: Deep Q-Learning](https://huggingface.co/learn/deep-rl-course/unit3/introduction) — same material, more diagrams.
- [GeeksforGeeks — Deep Q-Learning](https://www.geeksforgeeks.org/deep-q-learning/)
- [CleanRL's `dqn.py`](https://docs.cleanrl.dev/rl-algorithms/dqn/) — a clean, single-file reference implementation worth comparing to your own.
- Look up Double DQN. The max operator in our target introduces a systematic bias — what is it, and how does Double DQN fix it with essentially a one-line change? You'll implement this in the challenges section.

## 5.1 The Q-network

### Solution 2

Notice there's no activation on the output layer. Q-values are unbounded real numbers — LunarLander rewards range from roughly -100 for a crash to +200 for a good landing, so returns are negative for much of early training. A ReLU or sigmoid on the output would make negative Q-values unrepresentable, and the agent could never learn that crashing is bad.

In [ ]:
class QNetwork(nn.Module):
    def __init__(self, state_dim, n_actions, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),    nn.ReLU(),
            nn.Linear(hidden, n_actions),   # <- LINEAR output. No activation.
        )

    def forward(self, state):
        return self.net(state)

In [ ]:
# Sanity check — always shape-check a new network before training on it.
_net = QNetwork(8, 4).to(DEVICE)
_x = torch.randn(5, 8).to(DEVICE)
print("output shape:", _net(_x).shape, "  (expected torch.Size([5, 4]))")
print("params:", sum(p.numel() for p in _net.parameters()))
assert _net(_x).shape == (5, 4)

## 5.2 The replay buffer

### Solution 3

In [ ]:
Transition = namedtuple("Transition", ["state", "action", "reward", "next_state", "done"])

class ReplayBuffer:
    def __init__(self, capacity=100_000, batch_size=64):
        self.memory = deque(maxlen=capacity)
        self.batch_size = batch_size

    def add(self, state, action, reward, next_state, done):
        self.memory.append(Transition(state, action, reward, next_state, done))

    def sample(self):
        batch = random.sample(self.memory, k=self.batch_size)

        states      = torch.from_numpy(np.vstack([t.state      for t in batch])).float().to(DEVICE)
        actions     = torch.from_numpy(np.vstack([t.action     for t in batch])).long().to(DEVICE)
        rewards     = torch.from_numpy(np.vstack([t.reward     for t in batch])).float().to(DEVICE)
        next_states = torch.from_numpy(np.vstack([t.next_state for t in batch])).float().to(DEVICE)
        dones       = torch.from_numpy(np.vstack([float(t.done) for t in batch])).float().to(DEVICE)

        return states, actions, rewards, next_states, dones

    def __len__(self):
        return len(self.memory)

In [ ]:
# Sanity check the buffer.
_buf = ReplayBuffer(capacity=1000, batch_size=4)
for _ in range(50):
    _buf.add(np.random.randn(8).astype(np.float32), np.random.randint(4),
             np.random.randn(), np.random.randn(8).astype(np.float32), False)
_s, _a, _r, _ns, _d = _buf.sample()
for name, t in zip(["states", "actions", "rewards", "next_states", "dones"], [_s, _a, _r, _ns, _d]):
    print(f"{name:12s} {str(tuple(t.shape)):10s} {t.dtype}")
assert _s.shape == (4, 8) and _a.shape == (4, 1) and _a.dtype == torch.int64
print("\nBuffer OK.")

## 5.3 The agent — where the Bellman equation becomes code

### Solution 4

Every line that matters is commented below. Pay particular attention to `detach()`, `gather()`, and the `(1 - dones)` term — these are the three details that most implementations get subtly wrong.

In [ ]:
class DQNAgent:
    def __init__(self, state_dim, n_actions, lr=5e-4, gamma=0.99, tau=1e-3,
                 buffer_size=100_000, batch_size=64, update_every=4):
        self.n_actions   = n_actions
        self.gamma       = gamma
        self.tau         = tau
        self.update_every = update_every

        self.qnet_local  = QNetwork(state_dim, n_actions).to(DEVICE)
        self.qnet_target = QNetwork(state_dim, n_actions).to(DEVICE)
        self.qnet_target.load_state_dict(self.qnet_local.state_dict())   # start identical

        self.optimizer = torch.optim.Adam(self.qnet_local.parameters(), lr=lr)
        self.memory    = ReplayBuffer(buffer_size, batch_size)
        self.t_step    = 0

    def step(self, state, action, reward, next_state, done):
        self.memory.add(state, action, reward, next_state, done)
        self.t_step = (self.t_step + 1) % self.update_every
        if self.t_step == 0 and len(self.memory) > self.memory.batch_size:
            self.learn(self.memory.sample())

    def act(self, state, eps=0.0):
        if random.random() < eps:
            return random.randrange(self.n_actions)

        state_t = torch.from_numpy(state).float().unsqueeze(0).to(DEVICE)  # (1, 8)
        self.qnet_local.eval()                    # disables dropout/batchnorm if present
        with torch.no_grad():                     # no graph needed: we're not learning here
            action_values = self.qnet_local(state_t)
        self.qnet_local.train()
        return int(action_values.argmax(dim=1).item())

    def learn(self, experiences):
        states, actions, rewards, next_states, dones = experiences

        # --- 1. Target network's estimate of the best next-state value ---------
        # .max(1) returns (values, indices); [0] takes values; unsqueeze -> (B, 1)
        # .detach() is ESSENTIAL: this is a label, gradients must not flow through it.
        Q_targets_next = self.qnet_target(next_states).detach().max(1)[0].unsqueeze(1)

        # --- 2. Bellman target -------------------------------------------------
        # (1 - dones) zeroes the bootstrap term when the episode ended: there is
        # no s', so the return is exactly `reward`.
        Q_targets = rewards + self.gamma * Q_targets_next * (1 - dones)

        # --- 3. Current estimate for the action we ACTUALLY took ----------------
        # qnet_local(states) is (B, 4). We only observed the outcome of one action,
        # so gather() plucks out that column. Only it receives a gradient.
        Q_expected = self.qnet_local(states).gather(1, actions)

        # --- 4. Regression step ------------------------------------------------
        loss = F.mse_loss(Q_expected, Q_targets)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        self.soft_update()

    def soft_update(self):
        # Blend a tiny fraction of the local weights into the target network every
        # step, instead of a hard copy every N steps. Smoother, works well here.
        for target_p, local_p in zip(self.qnet_target.parameters(), self.qnet_local.parameters()):
            target_p.data.copy_(self.tau * local_p.data + (1.0 - self.tau) * target_p.data)

---
# Part 6 — Train it

## 6.1 The training loop

One subtlety worth pausing on: which flag do we store as `done` for learning purposes?

```python
done_for_learning = terminated                 # correct
done_for_learning = terminated or truncated    # subtly wrong
```

`truncated` means we hit the 1000-step limit while the lander was still flying — the future still has value. Marking that as terminal tells the agent "the world ends here, expect nothing more," which teaches it that hovering forever is worthless, but for the wrong reason.

In LunarLander the practical impact is small. In other environments it isn't. Get this right by habit.

**Question.** The loop below decays ε multiplicatively per episode, not per step, and floors at 0.01 — so even a fully trained agent still acts randomly 1% of the time. Why keep any exploration at all once the policy is good? What would you change if you wanted to maximize evaluation score instead?

In [ ]:
def train_dqn(agent, env, n_episodes=1200, max_t=1000,
              eps_start=1.0, eps_end=0.01, eps_decay=0.995,
              solve_score=200.0, verbose=True):
    scores, window = [], deque(maxlen=100)
    eps = eps_start
    t0 = time.time()

    for ep in range(1, n_episodes + 1):
        state, _ = env.reset()
        score = 0.0

        for _ in range(max_t):
            action = agent.act(state, eps)
            next_state, reward, terminated, truncated, _ = env.step(action)

            # `terminated` only -- see the discussion above.
            agent.step(state, action, reward, next_state, terminated)

            state = next_state
            score += reward
            if terminated or truncated:
                break

        scores.append(score); window.append(score)
        eps = max(eps_end, eps_decay * eps)

        if verbose and ep % 50 == 0:
            print(f"ep {ep:4d} | avg100 {np.mean(window):7.1f} | eps {eps:.3f} "
                  f"| {time.time()-t0:5.0f}s")

        if len(window) == 100 and np.mean(window) >= solve_score:
            print(f"\n SOLVED in {ep} episodes! avg100 = {np.mean(window):.1f} "
                  f"({time.time()-t0:.0f}s)")
            break

    return scores

## 6.2 Run it

This takes roughly 10-20 minutes. Expect the reward curve to dip before it rises: early on, the agent learns that firing engines costs fuel before it learns that landing pays off. That dip is normal and worth being able to recognize.

While it trains, go back to any "Look it up" sections you skipped, and start reading through the challenges in Part 7.

In [ ]:
set_seed(42)
env = gym.make("LunarLander-v3")
agent = DQNAgent(state_dim=8, n_actions=4)

scores = train_dqn(agent, env, n_episodes=1200)
env.close()

torch.save(agent.qnet_local.state_dict(), "dqn_lunarlander.pth")
print("Weights saved to dqn_lunarlander.pth")

In [ ]:
plot_scores(scores, title="DQN on LunarLander-v3")

## 6.3 Watch your agent fly

In [ ]:
agent.qnet_local.eval()
total = record_episode("LunarLander-v3", lambda o: agent.act(o, eps=0.0), "trained.mp4")
print(f"Episode reward: {total:.1f}")

In [ ]:
# Proper evaluation: 20 episodes, greedy policy, no learning.
env_eval = gym.make("LunarLander-v3")
eval_scores = []
for ep in range(20):
    state, _ = env_eval.reset(seed=1000 + ep)
    total, done = 0.0, False
    while not done:
        state, r, term, trunc, _ = env_eval.step(agent.act(state, eps=0.0))
        total += r
        done = term or trunc
    eval_scores.append(total)
env_eval.close()

print(f"Trained agent: mean {np.mean(eval_scores):7.1f} +/- {np.std(eval_scores):.1f}")
print(f"Random agent:  mean {RANDOM_BASELINE:7.1f}  (from Part 2)")
print(f"Episodes above 200: {sum(s >= 200 for s in eval_scores)}/20")

## 6.4 Look inside the agent's reasoning

Q-values are interpretable on their own. Let's check what the agent believes about a few specific situations, and see whether its reasoning lines up with ours.

In [ ]:
action_names = ["do nothing", "fire LEFT engine", "fire MAIN engine", "fire RIGHT engine"]

def inspect(state, label):
    s = torch.from_numpy(np.array(state, dtype=np.float32)).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        q = agent.qnet_local(s).cpu().numpy()[0]
    print(f"\n{label}")
    for i, (n, v) in enumerate(zip(action_names, q)):
        star = "  <-- chosen" if i == int(np.argmax(q)) else ""
        print(f"   {n:20s} Q = {v:8.2f}{star}")

# [x, y, vx, vy, angle, angular_vel, leg1, leg2]
inspect([0.0,  1.4,  0.0, -0.9, 0.0,  0.0, 0, 0], "Falling fast, centred, high up:")
inspect([0.0,  0.1,  0.0, -0.1, 0.0,  0.0, 0, 0], "Just above the pad, gentle descent:")
inspect([0.6,  1.0, -0.3,  0.0, 0.0,  0.0, 0, 0], "Far RIGHT of the pad, drifting left:")
inspect([0.0,  0.8,  0.0, -0.2, 0.6,  0.4, 0, 0], "Tilted hard and rotating:")
inspect([0.0,  0.0,  0.0,  0.0, 0.0,  0.0, 1, 1], "Landed, both legs down:")

**Question — look closely at what you just printed.**

1. In the "falling fast" case, did it choose to fire the main engine? Does the margin between the top two Q-values look confident, or nearly tied?
2. Compare the absolute Q-values across situations. "Landed, both legs down" should have a high value, and "tilted hard and rotating" a low one — does it? What does the magnitude of Q actually mean here?
3. In the "far right, drifting left" case, is the choice one you'd make yourself? If not, is the agent wrong, or are you? It optimized the actual reward function — did you, when you guessed?
4. Some of these states may be ones the agent rarely visited during training. Its Q-values there are extrapolation. Which of the five do you think are off-distribution, and how could you check?

---
# Part 7 — Challenges

Pick these based on the time you have. Each one is a real thing practitioners do.

---
### Challenge 1 (do this one) — Break it on purpose

Understanding why each piece exists is worth more than having it work. Pick two from the table below, and predict the outcome before running each one.

| Sabotage | How | Prediction, then verify |
|---|---|---|
| No target network | In `learn`, use `qnet_local` for `Q_targets_next` | |
| No replay | Set `batch_size=1` and sample only the most recent transition | |
| No exploration | `eps_start=0.0` | |
| Forget `.detach()` | Remove it | |
| γ too low | `gamma=0.5` | |
| Learning rate too high | `lr=1e-2` | |

Run 300 episodes for each case — enough to see the trend. Write down what the reward curve looks like each time. Being able to diagnose a broken RL run from its curve alone is the actual skill here.

---
### Challenge 2 (moderate) — Double DQN

Standard DQN uses the same network to both select and evaluate the best next action:

```python
Q_targets_next = self.qnet_target(next_states).detach().max(1)[0].unsqueeze(1)
```

Because of the max, any noise in the estimates gets systematically selected for. This is called maximization bias, and it makes DQN overestimate Q-values.

[Double DQN](https://arxiv.org/abs/1509.06461) decouples the two steps: select the action using the local network, and evaluate it using the target network. The change is about three lines of code.

Read the paper's abstract, work out the change yourself, and implement it. Does it train faster? Are the Q-values it produces lower, as the paper suggests they should be?

---
### Challenge 3 (moderate) — Hyperparameter study

Pick one of `lr`, `gamma`, `tau`, `batch_size`, `eps_decay`, or hidden layer size. Run 3-4 values for 400 episodes each, and plot them on one axis. Which one matters most?

Bonus: run the same configuration with three different seeds. RL has notoriously high seed variance — is your "improvement" actually bigger than the noise? ["Deep RL That Matters"](https://arxiv.org/abs/1709.06560) is worth reading on why single-seed results are close to meaningless.

---
### Challenge 4 (hard) — Continuous LunarLander

`gym.make("LunarLander-v3", continuous=True)` gives you a 2-dimensional continuous action space.

Why can't DQN handle this at all? Where exactly does `argmax_a Q(s,a)` break down?

Then look up DDPG, TD3, or SAC — the value-based family's answer to continuous action spaces. What do they add to make that argmax tractable again?

---
### Challenge 5 (hard) — A different family of algorithms

DQN learns a value function and derives a policy from it. Policy gradient methods optimize the policy directly instead.

Implement REINFORCE on LunarLander. Useful references:
[HF Unit 4](https://huggingface.co/learn/deep-rl-course/unit4/introduction),
[Karpathy's *Pong from Pixels*](https://karpathy.github.io/2016/05/31/rl/),
[Spinning Up: Intro to Policy Optimization](https://spinningup.openai.com/en/latest/spinningup/rl_intro3.html).

Compare the two approaches: which is more sample-efficient? Which is more stable? And why is PPO, a policy-gradient method, what's actually used in production settings, including for [RLHF on language models](https://huggingface.co/blog/rlhf)?

---
### Challenge 6 (do this one) — Publish it

Push your trained agent to the Hugging Face Hub and see how it stacks up:
[Deep RL Course Leaderboard](https://huggingface.co/spaces/huggingface-projects/Deep-RL-Course-Leaderboard),
[instructions for publishing](https://huggingface.co/learn/deep-rl-course/unit1/hands-on).

---
# Part 8 — Debugging RL (read this before your next project)

RL fails silently. Your code runs, the loss goes down, and the agent is still useless. A short checklist:

**Before blaming the algorithm**
- Did you check against a random baseline? Is your agent actually above it?
- Did you print the shape of every tensor inside `learn()`? A `(64,1)` versus `(64,)` mismatch broadcasts to `(64,64)` silently and quietly destroys your loss. This is the single most common DQN bug.
- Did you run three seeds? One bad seed proves nothing either way.

**Reading the reward curve**
- Flat at random level: not learning at all. Check `.detach()`, check `gather`, check the learning rate.
- Rises then collapses: instability. The target network is updating too fast, the learning rate is too high, or Q-values are diverging — print them and check.
- Rises then plateaus below solved: exploration ended too early, or capacity/γ is too low.
- Wildly noisy but trending upward: probably fine. RL training curves are just noisy.

Instrument rather than guess. Log the mean Q-value, TD loss, and ε alongside reward. Q-values diverging toward something like 1e4 tells you immediately that you're bootstrapping incorrectly.

[Andy Jones — Debugging RL, Without the Agonizing Pain](https://andyljones.com/posts/rl-debugging.html) is the best writeup on this topic. Worth bookmarking.

---
# Where to go from here

**Courses**
- [Hugging Face Deep RL Course](https://huggingface.co/learn/deep-rl-course/unit0/introduction) — free, hands-on, includes a certificate. Start here.
- [OpenAI Spinning Up](https://spinningup.openai.com/) — the best writing on policy gradients available.
- [David Silver's RL Course (DeepMind/UCL)](https://www.davidsilver.uk/teaching/) — the classic lecture series.
- [Sutton & Barto](http://incompleteideas.net/book/the-book-2nd.html) — the standard textbook, free.

**Code worth reading**
- [CleanRL](https://docs.cleanrl.dev/) — single-file, minimal abstraction, benchmarked. Read `dqn.py`, then `ppo.py`.
- [Stable-Baselines3](https://stable-baselines3.readthedocs.io/) — what you'd actually reach for on real work.
- [Gymnasium](https://gymnasium.farama.org/), [PettingZoo](https://pettingzoo.farama.org/) (multi-agent), [MiniGrid](https://minigrid.farama.org/).

**Papers, roughly in order**
1. [Playing Atari with Deep RL (2013)](https://arxiv.org/abs/1312.5602) — the original DQN paper.
2. [Human-level control (Nature, 2015)](https://www.nature.com/articles/nature14236) — introduces target networks.
3. [Double DQN (2015)](https://arxiv.org/abs/1509.06461), [Dueling DQN (2015)](https://arxiv.org/abs/1511.06581), [Prioritized Replay (2015)](https://arxiv.org/abs/1511.05952).
4. [Rainbow (2017)](https://arxiv.org/abs/1710.02298) — combines all of the above.
5. [PPO (2017)](https://arxiv.org/abs/1707.06347) — what most people actually use today.
6. [InstructGPT (2022)](https://arxiv.org/abs/2203.02155) — RL from human feedback, and how chat models get aligned.

**Quick references**
- [GeeksforGeeks — RL topic index](https://www.geeksforgeeks.org/what-is-reinforcement-learning/)
- [Lilian Weng's blog](https://lilianweng.github.io/) — policy gradients, exploration, RLHF.
- [r/reinforcementlearning](https://www.reddit.com/r/reinforcementlearning/)

---

### One last thing

You just built the algorithm that, scaled up, learned to play Atari from raw pixels in 2015 and helped start the deep RL era. The version in the Nature paper is essentially this code plus a convolutional front end and a lot more compute.

The gap between a toy notebook and an actual research result in RL is usually smaller than it looks from the outside. Go find out where it isn't.